In [1]:
# ============================================================
# 04_dec.py — DEC Fine-tuning on GPU
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import (
    DataLoader, TensorDataset)
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')
import os
import json

PROJECT_DIR = '/rds/homes/j/jxt554'
DATA_DIR    = f'{PROJECT_DIR}/data'
TABLES_DIR  = f'{PROJECT_DIR}/tables'
SEED        = 42
DEVICE      = torch.device('cuda')
K           = 2

torch.manual_seed(SEED)
np.random.seed(SEED)

print('DEC FINE-TUNING — GPU')
print('='*55)
print(f'Device: {torch.cuda.get_device_name(0)}')

X_cd = np.load(f'{DATA_DIR}/X_cd_int.npy')
X_uc = np.load(f'{DATA_DIR}/X_uc_int.npy')

print(f'CD: {X_cd.shape}')
print(f'UC: {X_uc.shape}')

# Best configs from grid search
BEST_CONFIGS = {
    'CD': {
        'AE'       : {'latent_dim':128,
                       'hidden':[256,128,64],
                       'dropout':0.3,'lr':0.0005,
                       'beta':1.0,
                       'model_type':'ae'},
        'DAE'      : {'latent_dim':64,
                       'hidden':[256,128,64],
                       'dropout':0.3,'lr':0.0005,
                       'beta':1.0,
                       'model_type':'dae'},
        'VAE'      : {'latent_dim':16,
                       'hidden':[256,128,64],
                       'dropout':0.3,'lr':0.001,
                       'beta':1.0,
                       'model_type':'vae'},
        'BetaVAE'  : {'latent_dim':32,
                       'hidden':[256,128,64],
                       'dropout':0.3,'lr':0.001,
                       'beta':4.0,
                       'model_type':'beta_vae'},
        'BatchVAE' : {'latent_dim':16,
                       'hidden':[256,128,64],
                       'dropout':0.3,'lr':0.001,
                       'beta':2.0,
                       'model_type':'batch_vae'},
    },
    'UC': {
        'AE'       : {'latent_dim':128,
                       'hidden':[512,256,128],
                       'dropout':0.3,'lr':0.001,
                       'beta':1.0,
                       'model_type':'ae'},
        'DAE'      : {'latent_dim':128,
                       'hidden':[256,128,64],
                       'dropout':0.3,'lr':0.0005,
                       'beta':1.0,
                       'model_type':'dae'},
        'VAE'      : {'latent_dim':128,
                       'hidden':[256,128,64],
                       'dropout':0.3,'lr':0.0005,
                       'beta':1.0,
                       'model_type':'vae'},
        'BetaVAE'  : {'latent_dim':128,
                       'hidden':[256,128,64],
                       'dropout':0.3,'lr':0.0005,
                       'beta':4.0,
                       'model_type':'beta_vae'},
        'BatchVAE' : {'latent_dim':16,
                       'hidden':[256,128,64],
                       'dropout':0.3,'lr':0.001,
                       'beta':1.0,
                       'model_type':'batch_vae'},
    }
}

DEC FINE-TUNING — GPU
Device: NVIDIA A100-SXM4-40GB
CD: (215, 991)
UC: (430, 991)


In [2]:
# ── Model definitions (same as before) ───────────
class Encoder(nn.Module):
    def __init__(self, input_dim,
                  latent_dim, hidden,
                  dropout):
        super().__init__()
        layers = []
        prev   = input_dim
        for h in hidden:
            layers += [nn.Linear(prev,h),
                       nn.BatchNorm1d(h),
                       nn.ReLU(),
                       nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev,latent_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

class Decoder(nn.Module):
    def __init__(self, latent_dim,
                  output_dim, hidden,
                  dropout):
        super().__init__()
        layers = []
        prev   = latent_dim
        for h in reversed(hidden):
            layers += [nn.Linear(prev,h),
                       nn.BatchNorm1d(h),
                       nn.ReLU(),
                       nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev,output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, z):
        return self.net(z)

def soft_assign(z, centroids, alpha=1.0):
    q = 1.0 / (1.0 + torch.sum(
        (z.unsqueeze(1) - centroids)**2,
        dim=2) / alpha)
    q = q**((alpha+1.0)/2.0)
    q = q / torch.sum(q, dim=1, keepdim=True)
    return q

def target_dist(q):
    p = q**2 / torch.sum(q, dim=0)
    p = p / torch.sum(p, dim=1, keepdim=True)
    return p

def kl_div(p, q):
    return torch.mean(torch.sum(
        p * torch.log(
            (p + 1e-8) / (q + 1e-8)),
        dim=1))

class DECModel(nn.Module):
    def __init__(self, encoder,
                  latent_dim, n_clusters):
        super().__init__()
        self.encoder   = encoder
        self.centroids = nn.Parameter(
            torch.randn(n_clusters,
                         latent_dim))

    def forward(self, x):
        z = self.encoder(x)
        q = soft_assign(z, self.centroids)
        return z, q

print('DEC model classes defined ✓')

DEC model classes defined ✓


In [3]:
# ── DEC training function ─────────────────────────
def run_dec(X, input_dim, latent_dim,
             hidden, dropout, lr,
             model_name, cohort,
             meta=None):

    print(f'\n  DEC: {model_name}')
    X_t = torch.FloatTensor(X).to(DEVICE)
    N   = len(X)

    # Step 1: Pretrain autoencoder
    encoder = Encoder(
        input_dim, latent_dim,
        hidden, dropout).to(DEVICE)
    decoder = Decoder(
        latent_dim, input_dim,
        hidden, dropout).to(DEVICE)

    opt_pre = torch.optim.Adam(
        list(encoder.parameters()) +
        list(decoder.parameters()),
        lr=lr)
    dl = DataLoader(
        TensorDataset(X_t),
        batch_size=64, shuffle=True)

    for ep in range(100):
        encoder.train(); decoder.train()
        for (xb,) in dl:
            opt_pre.zero_grad()
            loss = nn.functional.mse_loss(
                decoder(encoder(xb)), xb)
            loss.backward()
            opt_pre.step()

    # Step 2: Initialise centroids
    encoder.eval()
    with torch.no_grad():
        z_init = encoder(X_t).cpu().numpy()

    km = KMeans(n_clusters=K,
                 random_state=SEED,
                 n_init=20)
    km.fit(z_init)
    sil_before = silhouette_score(
        z_init, km.labels_)

    # Step 3: DEC fine-tuning
    dec = DECModel(
        encoder, latent_dim, K).to(DEVICE)
    dec.centroids.data = torch.FloatTensor(
        km.cluster_centers_).to(DEVICE)

    opt_dec = torch.optim.Adam(
        dec.parameters(), lr=1e-4)

    best_sil    = -1
    best_labels = None
    best_z      = None
    prev_labels = None

    for ep in range(200):
        dec.eval()
        with torch.no_grad():
            z_all, q_all = dec(X_t)
        p_all = target_dist(q_all).detach()

        dec.train()
        for i in range(0, N, 64):
            xb = X_t[i:i+64]
            pb = p_all[i:i+64]
            opt_dec.zero_grad()
            zb, qb = dec(xb)
            rb = decoder(zb)
            loss = (kl_div(pb, qb) +
                    0.1 * nn.functional.mse_loss(
                        rb, xb))
            loss.backward()
            opt_dec.step()

        if ep % 10 == 0:
            dec.eval()
            with torch.no_grad():
                z_cur, q_cur = dec(X_t)
            lbl_cur = q_cur.argmax(1).cpu().numpy()
            z_np    = z_cur.cpu().numpy()

            if len(np.unique(lbl_cur)) > 1:
                sil = silhouette_score(
                    z_np, lbl_cur)
                if sil > best_sil:
                    best_sil    = sil
                    best_labels = lbl_cur.copy()
                    best_z      = z_np.copy()

            if prev_labels is not None:
                delta = np.sum(
                    lbl_cur != prev_labels
                ) / N
                if delta < 0.001:
                    print(f'    Converged '
                          f'ep={ep}')
                    break
            prev_labels = lbl_cur.copy()

    print(f'    Before DEC: '
          f'{sil_before:.4f}')
    print(f'    After DEC : '
          f'{best_sil:.4f} '
          f'(+{best_sil-sil_before:.4f})')

    return best_z, best_labels, best_sil, sil_before

In [4]:
# ── Run DEC for all models ────────────────────────
def run_all_dec(X, meta, cohort):
    print(f'\n{"="*55}')
    print(f'{cohort} DEC FINE-TUNING')
    print(f'{"="*55}')

    input_dim = X.shape[1]
    configs   = BEST_CONFIGS[cohort]
    results   = []

    for model_name, cfg in configs.items():
        z, labels, sil_aft, sil_bef = run_dec(
            X,
            input_dim,
            cfg['latent_dim'],
            cfg['hidden'],
            cfg['dropout'],
            cfg['lr'],
            model_name,
            cohort)

        fname = model_name.lower().replace(
            '-','_').replace(' ','_')
        np.save(
            f'{DATA_DIR}/'
            f'latent_{cohort.lower()}'
            f'_{fname}_dec.npy', z)
        np.save(
            f'{DATA_DIR}/'
            f'labels_{cohort.lower()}'
            f'_{fname}_dec.npy', labels)

        results.append({
            'model'     : model_name,
            'sil_before': sil_bef,
            'sil_after' : sil_aft,
            'gain'      : sil_aft - sil_bef,
        })

    df = pd.DataFrame(results)
    df.to_csv(
        f'{TABLES_DIR}/'
        f'dec_results_{cohort.lower()}.csv',
        index=False)

    print(f'\n{cohort} DEC SUMMARY:')
    print(f'{"Model":<12} '
          f'{"Before":>8} '
          f'{"After":>8} '
          f'{"Gain":>8}')
    print('-'*40)
    for _, row in df.iterrows():
        print(f'{row["model"]:<12} '
              f'{row["sil_before"]:>8.4f} '
              f'{row["sil_after"]:>8.4f} '
              f'{row["gain"]:>8.4f}')

    return df

cd_dec_df = run_all_dec(X_cd, None, 'CD')
uc_dec_df = run_all_dec(X_uc, None, 'UC')

print('\nDEC COMPLETE')
print('Next: 05_consensus.py')


CD DEC FINE-TUNING

  DEC: AE
    Converged ep=40
    Before DEC: 0.3840
    After DEC : 0.6912 (+0.3072)

  DEC: DAE
    Converged ep=40
    Before DEC: 0.3930
    After DEC : 0.7037 (+0.3107)

  DEC: VAE
    Converged ep=20
    Before DEC: 0.3150
    After DEC : 0.4442 (+0.1292)

  DEC: BetaVAE
    Converged ep=70
    Before DEC: 0.3485
    After DEC : 0.7574 (+0.4089)

  DEC: BatchVAE
    Converged ep=70
    Before DEC: 0.3095
    After DEC : 0.7047 (+0.3952)

CD DEC SUMMARY:
Model          Before    After     Gain
----------------------------------------
AE             0.3840   0.6912   0.3072
DAE            0.3930   0.7037   0.3107
VAE            0.3150   0.4442   0.1292
BetaVAE        0.3485   0.7574   0.4089
BatchVAE       0.3095   0.7047   0.3952

UC DEC FINE-TUNING

  DEC: AE
    Converged ep=60
    Before DEC: 0.2519
    After DEC : 0.8319 (+0.5800)

  DEC: DAE
    Converged ep=50
    Before DEC: 0.3857
    After DEC : 0.7905 (+0.4049)

  DEC: VAE
    Converged ep=50
    Bef